Notebook Imports

In [1]:
# installing gdown to access online files
! pip install gdown

#installing pdfplumber to manage PDF enforcement sources
! pip install pdfplumber

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\hilla\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\hilla\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
#imports
import os
import json
import gdown
import shutil
import random
import sqlite3
import requests
import pdfplumber
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta



### **<br><u>Data Ingestion</u></br>**

Paths and Directories

In [3]:
# create and/or clean data landing zone
LANDING_DIR = Path.cwd().parent.joinpath('storage/landing_zone')
LANDING_DIR.mkdir(parents=True, exist_ok=True)
shutil.rmtree(LANDING_DIR, ignore_errors=True)

# sources directory
SOURCES = Path.cwd().parent.joinpath('sources/')

# storage directory
STORAGE = Path.cwd().parent.joinpath('storage/')

#### Operations data

In [4]:
# This data is synthetically generated data to mimick the internal operational
# data systems of the U.S. Immigration and Customs Enforcement (ICE)

In [5]:
random.seed(42) # allows for reproducible code

# Ensure the LANDING_DIR exists, as it might have been removed by a previous cell.
# This line is added to address the FileNotFoundError that occurs when OPS_DIR's parent is missing.
if not LANDING_DIR.is_dir():
    LANDING_DIR.mkdir(parents=True, exist_ok=True)

OPS_DIR = LANDING_DIR / "operations"
OPS_DIR.mkdir(exist_ok=True)

# random reproducable generation
def random_date(start_year=2020, end_year=2024):
  start = datetime(start_year, 1, 1)
  end = datetime(end_year, 12, 31)
  delta = end - start
  random_days = random.randint(0, delta.days)
  return start + timedelta(days=random_days)

In [6]:
# Defining Ice Departments

departments = ["Enforcement and Removal Operations (ERO)",
               "Homeland Security Investions (HSI)",
               "Homeland Security Investigations (HSI)",
               "Management & Administration",
               "Office of Professional Responsibility"]

# Defining Ice Unites
unites = [
    "Fugitive Operations",
    "Detention Management",
    "Transportation and Removal Operations",
    "Field Operations",
    "Intelligence"
]

# Detention Centers
detention_centers = [
    "Stewart Detention Center (GA)",
    "Adelanto ICE Processing Center (CA)",
    "Otay Mesa Detention Center (CA)",
    "Eloy Detention Center (AZ)",
    "Tacoma ICE Processing Center (WA)",
    "Aurora ICE Processing Center (CO)",
    "Port Isabel Detention Center (TX)",
    "Krome Detention Center (FL)",
    "LaSalle ICE Processing Center (LA)",
    "Florence Detention Center (AZ)"
]

#Departments
department_codes = [
    "015", "020", "025", "030", "045", "050", "060", "070", "080", "090"
    ]

# Regions
regions = [
    "Southeast",
    "West",
    "Southwest",
    "Mountain"
]

# Roles
roles = ["Agent", "Senior Agent", "Supervisor", "Deputy Director", "Analyst"]

In [7]:
# generating 400 Rows: Source Code Gemini
num_records = 400
data = []

for agent_id in range(1, num_records + 1):
  region = random.choice(regions)
  role = random.choice(roles)
  department = random.choice(departments)
  unit = random.choice(unites)
  detention_center = random.choice(detention_centers)

  start_date = random_date()

# creating 30% chance end date is null
  if random.random() < 0.3:
    end_date = None
  else:
    end_date = start_date + timedelta(days=random.randint(30,1500))

# supervisor_id: random integer or null
  supervisor_id = random.choice([None] + list(range(1, num_records + 1)))

  data.append({
    "agent_id": agent_id,
    "region": region,
    "role": role,
    "department": department,
    "unit": unit,
    "assignment_start_date": start_date.strftime("%Y-%m-%d"),
    "assignment_end_date": None if end_date is None else end_date.strftime("%Y-%m-%d"),
    "detention_center_name": detention_center,
    "supervisor_id": supervisor_id})
ice_db = pd.DataFrame(data)
ice_db.head()

,agent_id,region,role,department,unit,assignment_start_date,assignment_end_date,detention_center_name,supervisor_id
0,1,Southeast,Agent,Homeland Security Investigations (HSI),Detention Management,2020-10-12,2024-08-27,Eloy Detention Center (AZ),379.0
1,2,Southeast,Analyst,Management & Administration,Fugitive Operations,2020-07-10,NaN,Stewart Detention Center (GA),258.0
2,3,Southeast,Analyst,Homeland Security Investions (HSI),Intelligence,2021-03-27,2022-11-16,Port Isabel Detention Center (TX),3.0
3,4,West,Deputy Director,Homeland Security Investigations (HSI),Transportation and Removal Operations,2021-03-16,2023-03-05,Otay Mesa Detention Center (CA),52.0
4,5,Southeast,Deputy Director,Enforcement and Removal Operations (ERO),Transportation and Removal Operations,2023-05-21,NaN,Aurora ICE Processing Center (CO),22.0


In [8]:
# Saving syntheic Operations Dataset
ice_db.to_csv(OPS_DIR / "ice_operations.csv", index=False)
print("Saved synthetic operations dataset as ice_db.csv")

Saved synthetic operations dataset as ice_db.csv


In [9]:
ice_db.info()

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   agent_id               400 non-null    int64  
 1   region                 400 non-null    str    
 2   role                   400 non-null    str    
 3   department             400 non-null    str    
 4   unit                   400 non-null    str    
 5   assignment_start_date  400 non-null    str    
 6   assignment_end_date    283 non-null    str    
 7   detention_center_name  400 non-null    str    
 8   supervisor_id          398 non-null    float64
dtypes: float64(1), int64(1), str(7)
memory usage: 28.3 KB


In [10]:
# Added by Sonali to check whether agent_id can serve as a primary key:

print("Duplicate agent_id values:", ice_db["agent_id"].duplicated().sum())
print("Null agent_id values:", ice_db["agent_id"].isna().sum())

Duplicate agent_id values: 0
Null agent_id values: 0


In [11]:
# Added by Sonali to check whether supervisor_id can reference agent_id as a self-referencing foreign key:

# The output of the following code block is set() which means the set is empty, so no invalid supervisor IDs were found.
# In other words, every non-null supervisor_id in ice_db also appears somewhere in the agent_id column.
# That supports this relationship: supervisor_id → agent_id
# So supervisor_id is a good candidate for a self-referencing foreign key within the same table.
# The two null supervisor_id values are also reasonable. They likely represent top-level supervisors who do not report to another agent in this dataset.

invalid_supervisors = set(
    ice_db["supervisor_id"].dropna().astype(int)
) - set(
    ice_db["agent_id"]
)

print("Invalid supervisor IDs:", invalid_supervisors)

Invalid supervisor IDs: set()


#### Financial Data

In [12]:
# This data is the U.S. Treasury's Fiscal Data retrieved via API.
# Additionally, this data has been augmented with LLM-generated ICE funding data

In [13]:
OPS_DIR = LANDING_DIR / "financial"
OPS_DIR.mkdir(exist_ok=True)

API_BASE_URL = "https://api.fiscaldata.treasury.gov/services/api/fiscal_service"

# All OD endpoints from the Treasury table
od_endpoints = [
    "/v1/accounting/od/statement_net_cost",
    "/v1/accounting/od/net_position",
    "/v1/accounting/od/reconciliations",
    "/v1/accounting/od/cash_balance",
    "/v1/accounting/od/balance_sheets",
    "/v1/accounting/od/long_term_projections",
    "/v1/accounting/od/social_insurance",
    "/v1/accounting/od/insurance_amounts"
]

all_tables = {}

for ep in od_endpoints:
    print(f"Pulling: {ep}")

    params = {"page[number]": 1, "page[size]": 500}
    response = requests.get(f"{API_BASE_URL}{ep}", params=params)

    if response.status_code == 200:
        df = pd.DataFrame(response.json().get("data", []))
        all_tables[ep] = df
        print(f"Rows retrieved: {len(df)}")
    else:
        print(f"Failed with status: {response.status_code}")


Pulling: /v1/accounting/od/statement_net_cost


Failed with status: 404
Pulling: /v1/accounting/od/net_position


Rows retrieved: 500
Pulling: /v1/accounting/od/reconciliations


Rows retrieved: 500
Pulling: /v1/accounting/od/cash_balance


Rows retrieved: 500
Pulling: /v1/accounting/od/balance_sheets


Failed with status: 404
Pulling: /v1/accounting/od/long_term_projections


Rows retrieved: 172
Pulling: /v1/accounting/od/social_insurance


Rows retrieved: 500
Pulling: /v1/accounting/od/insurance_amounts


Rows retrieved: 287


In [14]:
# ------------ Treasury Data API----------------

# Implementation Pagination Source Code: API & CSV Ingestion Patterns in Python

def fetch_all_treasury_data(endpoint, page_size=500, max_pages=50):
    all_data = []
    page_number = 1

    while True:
        if page_number > max_pages:
            print("WARNING: Reached max_pages safety limit")
            break

        url = f"{API_BASE_URL}{endpoint}"
        params = {
            "page[number]": page_number,
            "page[size]": page_size
        }

        response = requests.get(url, params=params)

        if response.status_code != 200:
            print(f"ERROR: Request failed with status {response.status_code}")
            break

        json_data = response.json()
        page_data = json_data.get("data", [])

        # Treasury returns empty list when done
        if not page_data:
            print("INFO: No more data returned – pagination complete")
            break

        all_data.extend(page_data)

        # Last page has fewer rows
        if len(page_data) < page_size:
            print("INFO: Last page reached")
            break

        page_number += 1

    return all_data

In [15]:
records = fetch_all_treasury_data("/v1/accounting/od/reconciliations")
print(len(records))


INFO: Last page reached
594


In [16]:
# Complete Treasury API Pipeline - Extract and Land

def extract_treasury_api_data(endpoint, output_prefix, page_size=500, max_pages=50):
    print(f"Extracting from Treasury endpoint: {endpoint}")

    all_data = fetch_all_treasury_data(endpoint, page_size, max_pages)

    if not all_data:
        print(f"ERROR:No data extracted")
        return None

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = OPS_DIR / f"{output_prefix}_{timestamp}.json"

    output_data = {
        "metadata": {
            "timestamp": datetime.now().isoformat(),
            "source": endpoint,
            "record_count": len(all_data),
            "extraction_status": "success"
        },
        "data": all_data
    }

    with open(output_file, "w") as f:
        json.dump(output_data, f, indent=2)

    print(f"INFO:Successfully saved {len(all_data)} records to {output_file}")
    return str(output_file)


output_path = extract_treasury_api_data(
    endpoint="/v1/accounting/od/reconciliations",
    output_prefix="treasury_reconciliations",
    page_size=500
)

print("Output path:", output_path)


Extracting from Treasury endpoint: /v1/accounting/od/reconciliations


INFO: Last page reached
INFO:Successfully saved 594 records to c:\Users\hilla\Desktop\Group_4_Clarity_Analytics_Center\storage\landing_zone\financial\treasury_reconciliations_20260814_134251.json
Output path: c:\Users\hilla\Desktop\Group_4_Clarity_Analytics_Center\storage\landing_zone\financial\treasury_reconciliations_20260814_134251.json


In [17]:
# Normalize Treasury JSON → DataFrame

def normalize_treasury_json(json_path):
    """
    Normalize landed Treasury JSON into a clean DataFrame.
    """

    print(f"Normalizing JSON file: {json_path}")

    with open(json_path, "r") as f:
        raw = json.load(f)

    data = raw.get("data", [])

    if not data:
        print("ERROR: No data found in JSON file")
        return None

    df = pd.json_normalize(data)

    print(f"INFO: Normalized DataFrame with {len(df)} rows and {len(df.columns)} columns")

    return df

if output_path:
    df_recon = normalize_treasury_json(output_path)
    df_recon.head()
else:
    print("Data extraction failed, cannot normalize Treasury JSON.")

Normalizing JSON file: c:\Users\hilla\Desktop\Group_4_Clarity_Analytics_Center\storage\landing_zone\financial\treasury_reconciliations_20260814_134251.json
INFO: Normalized DataFrame with 594 rows and 14 columns


In [18]:
df_recon.info()

<class 'pandas.DataFrame'>
RangeIndex: 594 entries, 0 to 593
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   record_date              594 non-null    str  
 1   stmt_fiscal_year         594 non-null    str  
 2   restmt_flag              594 non-null    str  
 3   account_desc             594 non-null    str  
 4   component_desc           594 non-null    str  
 5   line_item_desc           594 non-null    str  
 6   position_bil_amt         594 non-null    str  
 7   src_line_nbr             594 non-null    str  
 8   record_fiscal_year       594 non-null    str  
 9   record_fiscal_quarter    594 non-null    str  
 10  record_calendar_year     594 non-null    str  
 11  record_calendar_quarter  594 non-null    str  
 12  record_calendar_month    594 non-null    str  
 13  record_calendar_day      594 non-null    str  
dtypes: str(14)
memory usage: 65.1 KB


In [19]:
# ------------ Synthetic ICE financial data ------------
# Gemini used to support the base code


def generate_synthetic_ice_budget(num_records=400):
    """
    Generate synthetic ICE budget data aligned with Treasury reconciliation fields.

    Args:
        num_records (int): Number of synthetic rows to generate.

    Returns:
        pd.DataFrame: Synthetic ICE budget dataset.
    """

    print(f"Info:Generating {num_records} synthetic ICE budget records")

    # Fiscal years aligned with Treasury dataset
    fiscal_years = np.random.choice(range(2020, 2025), size=num_records)

    # Synthetic department codes (Treasury uses 2–3 digit codes)
    department_codes = np.random.choice(
        ["015", "020", "025", "030", "045", "050", "060", "070", "080", "090"],
        size=num_records
    )

    # Synthetic agency names
    agencies = [
        "Immigration and Customs Enforcement",
        "Customs and Border Protection",
        "Department of Homeland Security",
        "Federal Protective Service",
        "Office of Biometric Identity Management"
    ]
    agency_names = np.random.choice(agencies, size=num_records)

    # Synthetic budget fields (realistic ranges)
    net_operating_cost = np.random.uniform(1e8, 5e9, size=num_records)  # $100M–$5B
    budget_authority = np.random.uniform(5e8, 7e9, size=num_records)    # $500M–$7B
    budget_deficit_contribution = net_operating_cost * np.random.uniform(0.8, 1.2, size=num_records)

    # Build DataFrame
    df = pd.DataFrame({
        "fiscal_year": fiscal_years,
        "department_code": department_codes,
        "agency_name": agency_names,
        "net_operating_cost": net_operating_cost.round(2),
        "budget_authority": budget_authority.round(2),
        "budget_deficit_contribution": budget_deficit_contribution.round(2),
        "source": "synthetic_ICE"
    })

    print(f"INFO:Synthetic ICE dataset generated successfully")
    return df


# Saving Data
def save_synthetic_ice(df, output_prefix="synthetic_ice"):
    """
    Save synthetic ICE dataset to landing zone with metadata wrapper.
    """

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    FIN_DIR = LANDING_DIR / "financial"
    FIN_DIR.mkdir(exist_ok=True)

    output_file = FIN_DIR / f"{output_prefix}_{timestamp}.json"

    output_data = {
        "metadata": {
            "timestamp": datetime.now().isoformat(),
            "source": "synthetic ICE generator",
            "record_count": len(df),
            "extraction_status": "success"
        },
        "data": df.to_dict(orient="records")
    }

    with open(output_file, "w") as f:
        json.dump(output_data, f, indent=2)

    print(f"INFO:Synthetic ICE data saved to {output_file}")
    return str(output_file)


In [20]:
ice_fiscaldata = generate_synthetic_ice_budget(num_records=300)
ice_path = save_synthetic_ice(ice_fiscaldata)

print("Synthetic ICE file saved at:", ice_path)


Info:Generating 300 synthetic ICE budget records
INFO:Synthetic ICE dataset generated successfully
INFO:Synthetic ICE data saved to c:\Users\hilla\Desktop\Group_4_Clarity_Analytics_Center\storage\landing_zone\financial\synthetic_ice_20260814_134251.json
Synthetic ICE file saved at: c:\Users\hilla\Desktop\Group_4_Clarity_Analytics_Center\storage\landing_zone\financial\synthetic_ice_20260814_134251.json


#### Enforcement data

In [21]:
# This data represents ICE Enforcement and Removal Operations reports in PDF format
# These PDFs are sourced from: https://www.ice.gov/statistics
# The years of the original reports have been altered to reflect prior datasets

In [22]:
ENFORCEMENT_DIR = LANDING_DIR / "enforcement_pdfs"
ENFORCEMENT_DIR.mkdir(parents=True, exist_ok=True)

pdf_sources = {
    "FY_2024_ICE_ERO_Report.pdf": "FY 2020 ICE ERO Report.pdf", #Original publication year: 2020
    "FY_2023_ICE_ERO_Report.pdf": "FY 2019 ICE ERO Report.pdf", #Original publication year: 2019
    "FY_2022_ICE_ERO_Report.pdf": "FY 2018 ICE ERO Report.pdf", #Original publication year: 2018
    "FY_2021_ICE_ERO_Report.pdf": "FY 2017 ICE ERO Report.pdf", #Original publication year: 2017
    "FY_2020_ICE_ERO_Report.pdf": "FY 2016 ICE ERO Report.pdf", #Original publication year:2016
}

for new_file_name, old_file_name in pdf_sources.items():
    
    old_file_path = SOURCES / old_file_name
    new_file_path = ENFORCEMENT_DIR / new_file_name

    if not old_file_path.exists():
        print(f"{old_file_name} does not exist in sources directory: {SOURCES}")

    if new_file_path.exists():
        print(f"{new_file_name} already exists in {STORAGE}. Skipping download.")
    else:
        shutil.copy(old_file_path, new_file_path)

In [23]:
def extract_pdf_raw(pdf_path, output_prefix="enforcement_pdf"):
    raw_text = []

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                raw_text.append({
                    "page_number": i + 1,
                    "content": page_text
                })
    except Exception as e:
        print(f"ERROR: PDF extraction failed from {pdf_path}: {e}")
        return None

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = LANDING_DIR / f"{output_prefix}_{timestamp}.json"

    output_data = {
        "metadata": {
            "timestamp": datetime.now().isoformat(),
            "source": str(pdf_path),
            "record_count": len(raw_text),
            "extraction_status": "success"
        },
        "data": raw_text
    }

    with open(output_file, "w") as f:
        json.dump(output_data, f, indent=2)

    print(f"Saved PDF extraction to {output_file}")
    return str(output_file)


In [24]:
# Added by Hillary: ERO ENFORCEMENT DATA.
# we would have to write extensively complex regex patterns to extract enforcement metrics
# out of the ERO PDFS, therefore we have used Claude Opus 5 to do so and added another bronze source
# bronze_ice_enforcement_metrics that will be joined with bronze_ice_enforcement_pdfs to populate silver_enforcement

enforcement_metrics_csv_name = "ero_enforcement_metrics_claude_extract.csv"

enforcement_metrics_csv_source_path = SOURCES / enforcement_metrics_csv_name
enforcement_metrics_csv_target_path = LANDING_DIR / enforcement_metrics_csv_name

if not enforcement_metrics_csv_source_path.exists():
    print(f"{enforcement_metrics_csv_name} does not exist in sources directory: {SOURCES}")

if enforcement_metrics_csv_target_path.exists():
    print(f"{enforcement_metrics_csv_name} already exists in {STORAGE}. Skipping download.")
else:
    shutil.copy(enforcement_metrics_csv_source_path, enforcement_metrics_csv_target_path)
    enforcement_metrics_df = pd.read_csv(enforcement_metrics_csv_target_path)

## Bronze Loading


In [25]:
USE_DRIVE = False

db_file_name = 'clarity_analytics_center.db'
db_path = Path(STORAGE).joinpath(db_file_name)

# Create or connect to the database
conn = sqlite3.connect(db_path)

In [26]:
print("Connected to database.")

tables = [
    "bronze_ice_operations",
    "bronze_ice_budget",
    "bronze_treasury_reconciliation",
    "bronze_ice_enforcement_pdfs",
    "bronze_ice_enforcement_metrics"
]


# 1. Truncate all tables
print("\nTruncate bronze tables")
for table in tables:
    conn.execute(f"DELETE FROM {table}")


# 2. Loading Operations -> bronze_ice_operations
print("Loading Opeations into Bronze...")

ice_db.to_sql(
    "bronze_ice_operations",
    conn,
    if_exists='append',
    index=False
)

print("Operations loaded into Bronze.")


# 3. Loading Syntetic ICE Budget into Bronze
print("Loading ICE Budget into Bronze...")

ice_fiscaldata.to_sql(
    "bronze_ice_budget",
    conn,
    if_exists='append',
    index=False)

print("ICE Budget loaded into Bronze.")

# 4. Loading Treasury Reconcilation into Bronze
print("Loading Treasury Reconciliation into Bronze...")

df_recon.to_sql(
    "bronze_treasury_reconciliation",
    conn,
    if_exists='append',
    index=False
)

# 5. Loading Enforcement PDFS and metrics into Bronze

enforcement_rows = []

for filename in pdf_sources.keys():
  pdf_path = ENFORCEMENT_DIR / filename
  extraction = extract_pdf_raw(pdf_path)

  # Loading jSON created by extract_pdf_raw
  with open(extraction, "r", encoding="utf-8") as f:
    payload = json.load(f)

  metadata = payload["metadata"]
  pages = payload["data"]

  for page in pages:
    enforcement_rows.append({
      "extraction_timestamp": metadata.get("timestamp"),
      "source_file": metadata.get("source"),
      "record_count": metadata.get("record_count"),
      "extraction_status": metadata.get("extraction_status"),
      "page_number": page.get("page_number"),
      "content": page.get("content")
  })
  enforcement_pdfs_df = pd.DataFrame(enforcement_rows)

enforcement_pdfs_df.to_sql(
    "bronze_ice_enforcement_pdfs",
    conn,
    if_exists='append',
    index=False
)
print("Enforcement PDFs loaded into Bronze.")

enforcement_metrics_df.to_sql(
    "bronze_ice_enforcement_metrics",
    conn,
    if_exists='append',
    index=False
)
print("Enforcement Metrics loaded into Bronze.")


# 6. Validation
print("\nBronze Load Summary:")
for table in tables:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count} rows")



Connected to database.

Truncate bronze tables
Loading Opeations into Bronze...
Operations loaded into Bronze.
Loading ICE Budget into Bronze...
ICE Budget loaded into Bronze.
Loading Treasury Reconciliation into Bronze...


Saved PDF extraction to c:\Users\hilla\Desktop\Group_4_Clarity_Analytics_Center\storage\landing_zone\enforcement_pdf_20260814_134257.json


Saved PDF extraction to c:\Users\hilla\Desktop\Group_4_Clarity_Analytics_Center\storage\landing_zone\enforcement_pdf_20260814_134302.json


Saved PDF extraction to c:\Users\hilla\Desktop\Group_4_Clarity_Analytics_Center\storage\landing_zone\enforcement_pdf_20260814_134304.json


Saved PDF extraction to c:\Users\hilla\Desktop\Group_4_Clarity_Analytics_Center\storage\landing_zone\enforcement_pdf_20260814_134307.json


Saved PDF extraction to c:\Users\hilla\Desktop\Group_4_Clarity_Analytics_Center\storage\landing_zone\enforcement_pdf_20260814_134309.json
Enforcement PDFs loaded into Bronze.
Enforcement Metrics loaded into Bronze.

Bronze Load Summary:
bronze_ice_operations: 400 rows
bronze_ice_budget: 300 rows
bronze_treasury_reconciliation: 594 rows
bronze_ice_enforcement_pdfs: 125 rows
bronze_ice_enforcement_metrics: 20 rows


In [27]:
#Validating before closing connection
print("\nBronze Load Summary:")
for table in tables:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count} rows")

conn.commit()
conn.close()

print("\nBronze loaded. Connection Closed")


Bronze Load Summary:
bronze_ice_operations: 400 rows
bronze_ice_budget: 300 rows
bronze_treasury_reconciliation: 594 rows
bronze_ice_enforcement_pdfs: 125 rows
bronze_ice_enforcement_metrics: 20 rows

Bronze loaded. Connection Closed
